<a href="https://www.kaggle.com/code/virajyawale/notebook02556e33c6?scriptVersionId=308326824" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Fake News Detection — Model Usage Guide

> **Model:** `virajyawale/fakenewsdetection`  
> **Architecture:** Word2Vec + LSTM (Keras/TensorFlow)  
> **Test Accuracy:** 98.8%  
> **Author:** Viraj Yawale

This notebook shows how to load and use the Fake News Detection model from Kaggle.  
The model classifies any news text as **REAL** or **FAKE** using a deep learning LSTM network.

---

## What's inside the model?
| File | Purpose |
|---|---|
| `saved_model.keras` | Trained Keras LSTM model (~88 MB) |
| `tokenizer.pkl` | Keras Tokenizer — maps words to integers |

## Model Pipeline
```
Raw Text → Clean → Tokenize → Pad (1000) → LSTM → Score (0–1) → REAL/FAKE
```

## Step 1 — Install & Import

In [ ]:
import numpy as np
import pickle
import re
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

print('TensorFlow and dependencies ready!')

## Step 2 — Load Model from Kaggle

The model files are attached via **Input** on the right panel.  
Path: `/kaggle/input/fakenewsdetection/`

In [ ]:
import os

# Check what files are available
BASE_PATH = '/kaggle/input/models/virajyawale/fakenewsdetection'

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f'{full_path}  ({size_mb:.1f} MB)')

In [ ]:
# Load the Keras LSTM model
print('Loading model... (may take 30–60 seconds)')
model = load_model(f'{BASE_PATH}/tensorflow2/default/2/saved_model.keras')

print('Model loaded!')
model.summary()

In [ ]:
# Load the Tokenizer
print('Loading tokenizer...')
with open(f'{BASE_PATH}tensorflow2/default/2/tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

print(f'Tokenizer loaded! Vocabulary size: {len(tokenizer.word_index):,} words')

## Step 3 — Prediction Function

In [ ]:
# Constants — must match training
MAXLEN    = 1000   # max tokens per article
THRESHOLD = 0.8    # score > 0.8 → REAL, else → FAKE

def clean_text(text):
    """Same cleaning used during training."""
    text = str(text).lower()
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)  # remove special characters
    text = re.sub(r'\s+', ' ', text).strip()     # remove extra whitespace
    return text

def predict_news(text):
    """
    Predict whether a news article is REAL or FAKE.
    
    Args:
        text (str): News article or headline text
    
    Returns:
        dict: label, confidence, raw_score
    """
    # 1. Clean
    cleaned = clean_text(text)
    
    # 2. Tokenize — convert words to integer indices
    seq = tokenizer.texts_to_sequences([cleaned])
    
    # 3. Pad — make length exactly 1000
    padded = pad_sequences(seq, maxlen=MAXLEN)
    
    # 4. Predict — get score between 0 and 1
    raw_score = float(model.predict(padded, verbose=0)[0][0])
    
    # 5. Apply threshold
    label = 'REAL' if raw_score > THRESHOLD else 'FAKE'
    confidence = round(
        (raw_score if label == 'REAL' else 1 - raw_score) * 100, 2
    )
    
    return {
        'label':      label,
        'confidence': f'{confidence}%',
        'raw_score':  round(raw_score, 4)
    }

print('predict_news() function ready!')

## Step 4 — Test with Sample News Articles

In [ ]:
# Test articles
test_cases = [
    {
        'text': 'WASHINGTON (Reuters) - The United States Senate passed a '
                'bipartisan infrastructure bill worth 1.2 trillion dollars, '
                'allocating funds for roads, bridges, broadband and clean energy.',
        'expected': 'REAL'
    },
    {
        'text': 'SHOCKING: Scientists confirm the moon is completely hollow and '
                'was placed by aliens 10000 years ago. NASA has been hiding this '
                'truth from the public for decades according to insider sources.',
        'expected': 'FAKE'
    },
    {
        'text': 'Inaugurating the new Pamban bridge in Rameswaram, PM Modi says '
                'it is the country first vertical lift railway sea bridge and a '
                'symbol of modern India infrastructure.',
        'expected': 'REAL'
    },
    {
        'text': 'BREAKING: Donald Trump secretly funded by George Soros according '
                'to leaked documents showing 500 million dollar transfer. '
                'The mainstream media is hiding this from you!',
        'expected': 'FAKE'
    },
    {
        'text': 'The Federal Reserve raised interest rates by 25 basis points on '
                'Wednesday as policymakers continued their fight against inflation, '
                'marking the tenth increase in the current tightening cycle.',
        'expected': 'REAL'
    }
]

print('=' * 60)
print(f'{"Article":<10} {"Expected":<10} {"Predicted":<10} {"Confidence":<12} {"Correct"}')
print('=' * 60)

correct = 0
for i, case in enumerate(test_cases, 1):
    result = predict_news(case['text'])
    is_correct = result['label'] == case['expected']
    if is_correct:
        correct += 1
    tick = 'YES' if is_correct else 'NO'
    print(f'{i:<10} {case["expected"]:<10} {result["label"]:<10} {result["confidence"]:<12} {tick}')

print('=' * 60)
print(f'Result: {correct}/{len(test_cases)} correct ({correct/len(test_cases)*100:.0f}%)')

## Step 5 — Try Your Own News Text

In [ ]:
# Paste any news article or headline here
my_news = """
India successfully launched its Chandrayaan-4 mission from the Satish Dhawan 
Space Centre in Sriharikota on Thursday. The mission aims to bring back lunar 
samples to Earth, making India only the fourth country to achieve this feat 
after the United States, Soviet Union, and China.
"""

result = predict_news(my_news)

print('━' * 40)
print(f'  Prediction : {result["label"]}')
print(f'  Confidence : {result["confidence"]}')
print(f'  Raw score  : {result["raw_score"]}  (0=FAKE ←→ 1=REAL)')
print('━' * 40)

## Step 6 — Batch Prediction on a List

In [ ]:
def predict_batch(texts):
    """
    Predict multiple news articles at once.
    More efficient than calling predict_news() in a loop.
    
    Args:
        texts (list): List of news article strings
    
    Returns:
        list: List of result dicts
    """
    cleaned = [clean_text(t) for t in texts]
    seqs    = tokenizer.texts_to_sequences(cleaned)
    padded  = pad_sequences(seqs, maxlen=MAXLEN)
    scores  = model.predict(padded, verbose=0).flatten()
    
    results = []
    for score in scores:
        label = 'REAL' if score > THRESHOLD else 'FAKE'
        conf  = round((score if label == 'REAL' else 1 - score) * 100, 2)
        results.append({'label': label, 'confidence': f'{conf}%', 'raw_score': round(float(score), 4)})
    return results


# Example batch
headlines = [
    'RBI keeps repo rate unchanged at 6.5 percent for sixth consecutive meeting.',
    'Bill Gates microchips found in COVID vaccines by independent researchers!',
    'Supreme Court upholds reservation policy in government job promotions.',
    'Obama born in Kenya — new documents prove massive cover-up by Democrats.',
]

batch_results = predict_batch(headlines)

print('Batch Predictions:')
print('─' * 70)
for headline, res in zip(headlines, batch_results):
    print(f'  [{res["label"]}] {res["confidence"]:>7}  —  {headline[:55]}...')
print('─' * 70)

---

## Understanding the Score

| Raw Score | Interpretation |
|---|---|
| 0.95 – 1.00 | Very confidently REAL |
| 0.80 – 0.95 | Likely REAL |
| 0.50 – 0.80 | Uncertain (borderline) |
| 0.20 – 0.50 | Likely FAKE |
| 0.00 – 0.20 | Very confidently FAKE |

## Model Architecture
```
Input text (any length)
    ↓  clean_text()
Cleaned text
    ↓  tokenizer.texts_to_sequences()
Integer sequence
    ↓  pad_sequences(maxlen=1000)
Fixed-length sequence [1000 integers]
    ↓  Embedding layer (Word2Vec, 100-dim, frozen)
Word vectors [1000 × 100]
    ↓  LSTM(128 units)
Context vector [128]
    ↓  Dense(1, sigmoid)
Score [0.0 → 1.0]
    ↓  Threshold 0.8
FAKE or REAL
```

---
*Model by Viraj Yawale | github.com/VirajYawale*